# 06 ? C2ST on Taiwan: comparable demarcation probe

Notebook 05 showed that SMOTE-synthetic churn minority points are clearly detectable by C2ST. This notebook asks whether the same point-level detectability appears on the cross-sectional Taiwan bankruptcy dataset, where notebook 03 found SMOTE to be neutral for ranking.

The important part is comparability: Taiwan has far fewer minority examples and more features than the churn C2ST task, so this notebook reports the full Taiwan result plus sample-size-matched and feature-count-matched controls. A high, low, or inconclusive result is scientifically valid; nothing here is tuned to move AUC.

## Setup

In [1]:
import hashlib
import inspect
import json
import platform
import re
import sys
from pathlib import Path

import imblearn
import matplotlib
import numpy as np
import pandas as pd
import scipy
import sklearn
from IPython.display import Markdown, display

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "README.md").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loaders import SIGNALS, load_churn_panel
from src.data.taiwan import download_taiwan, load_taiwan
from src.evaluation.c2st import (
    build_real_vs_duplicated_xy,
    build_real_vs_synthetic_xy,
    c2st_auc,
    c2st_importances,
)
import src.evaluation.c2st as c2st_module
from src.features.windows import WINDOW_PRESETS
from src.utils.config import OUTPUT_DIR, RAW_CHURN_PATH, SEED, set_global_seed

set_global_seed()

N_SPLITS = 5
N_REPEATS = 3
N_BOOT = 20
HONEST_MONTHS = WINDOW_PRESETS["HONEST"]
CHURN_FEATURE_COUNT = len(SIGNALS) * len(HONEST_MONTHS)

churn_df = load_churn_panel()
taiwan_path = download_taiwan()
X_taiwan, y_taiwan = load_taiwan(taiwan_path)

print(f"Churn rows: {len(churn_df):,}")
print(f"Taiwan rows/features: {len(X_taiwan):,} x {X_taiwan.shape[1]}")
print(f"Taiwan positives: {int(y_taiwan.sum()):,} ({y_taiwan.mean():.6f})")
print(f"Churn C2ST feature count: {CHURN_FEATURE_COUNT}")

Churn rows: 99,807
Taiwan rows/features: 6,819 x 95
Taiwan positives: 220 (0.032263)
Churn C2ST feature count: 40


## 1. Shared helpers

In [2]:
def _df_to_md_table(frame: pd.DataFrame, float_format: str = "{:.6f}") -> str:
    header = "| " + " | ".join([frame.index.name or ""] + list(frame.columns)) + " |"
    sep = "| " + " | ".join(["---"] * (len(frame.columns) + 1)) + " |"
    lines = [header, sep]
    for idx, row in frame.iterrows():
        row_cells = [str(idx)]
        for value in row:
            if isinstance(value, (float, np.floating)):
                row_cells.append(float_format.format(float(value)))
            else:
                row_cells.append(str(value))
        lines.append("| " + " | ".join(row_cells) + " |")
    return "\n".join(lines)


def _metadata(frame: pd.DataFrame) -> dict:
    return {k: v for k, v in frame.attrs.items() if not k.startswith("_")}


def _summary_from_values(values: list[float]) -> dict:
    arr = np.asarray(values, dtype=float)
    return {
        "mean_auc": float(arr.mean()),
        "std_auc": float(arr.std(ddof=1)) if len(arr) > 1 else 0.0,
        "ci_low": float(np.percentile(arr, 2.5)),
        "ci_high": float(np.percentile(arr, 97.5)),
        "n_runs": int(len(arr)),
    }


def _single_auc_summary(result: dict) -> dict:
    return {
        "mean_auc": result["mean"],
        "std_auc": result["std"],
        "ci_low": result["ci_low"],
        "ci_high": result["ci_high"],
        "n_runs": result["n_folds"],
    }


def _read_json_block_after_heading(report_text: str, heading: str) -> object:
    heading_idx = report_text.index(heading)
    remainder = report_text[heading_idx:]
    match = re.search(r"```json\n(.*?)\n```", remainder, re.DOTALL)
    if match is None:
        raise ValueError(f"No JSON block found after heading {heading!r}.")
    return json.loads(match.group(1))


def _churn_c2st_features(df: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray]:
    columns = [f"{signal}_{month}" for signal in SIGNALS for month in HONEST_MONTHS]
    X = df[columns].copy()
    for signal in ["TAG", "USAGE"]:
        signal_cols = [f"{signal}_{month}" for month in HONEST_MONTHS]
        X[signal_cols] = np.log1p(X[signal_cols])
    return X, df["CHURN"].to_numpy()


def _subsample_positive_class(
    X: pd.DataFrame, y: np.ndarray, n_positive: int, seed: int
) -> tuple[pd.DataFrame, np.ndarray]:
    rng = np.random.default_rng(seed)
    y_array = np.asarray(y)
    positive_idx = np.flatnonzero(y_array == 1)
    negative_idx = np.flatnonzero(y_array == 0)
    sampled_positive_idx = rng.choice(positive_idx, size=n_positive, replace=False)
    keep_idx = np.r_[negative_idx, sampled_positive_idx]
    rng.shuffle(keep_idx)
    return X.iloc[keep_idx].reset_index(drop=True), y_array[keep_idx]


def _run_task_pair(X: pd.DataFrame, y: np.ndarray, seed: int) -> dict:
    X_smote, y_smote, smote_meta = build_real_vs_synthetic_xy(X, y, seed=seed)
    X_dup, y_dup, dup_meta = build_real_vs_duplicated_xy(X, y, seed=seed)
    smote_auc = c2st_auc(X_smote, y_smote, n_splits=N_SPLITS, n_repeats=N_REPEATS, seed=seed)
    dup_auc = c2st_auc(X_dup, y_dup, n_splits=N_SPLITS, n_repeats=N_REPEATS, seed=seed)
    return {
        "smote_auc": smote_auc,
        "dup_auc": dup_auc,
        "smote_meta": smote_meta,
        "dup_meta": dup_meta,
        "task_balance": float(y_smote.mean()),
        "dup_balance": float(y_dup.mean()),
    }

## 2. Taiwan full C2ST

In [3]:
taiwan_full = _run_task_pair(X_taiwan, y_taiwan, seed=SEED)
taiwan_importances = c2st_importances(
    build_real_vs_synthetic_xy(X_taiwan, y_taiwan, seed=SEED)[0],
    build_real_vs_synthetic_xy(X_taiwan, y_taiwan, seed=SEED)[1],
    seed=SEED,
)
taiwan_top_importances = taiwan_importances.head(20).copy()

print(
    f"Taiwan full C2ST AUC: {taiwan_full['smote_auc']['mean']:.6f} +/- "
    f"{taiwan_full['smote_auc']['std']:.6f}"
)
print(f"Taiwan duplicated sanity AUC: {taiwan_full['dup_auc']['mean']:.6f}")
display(taiwan_top_importances)

Taiwan full C2ST AUC: 0.725551 +/- 0.058360
Taiwan duplicated sanity AUC: 0.500000


,feature,importance_mean,importance_std
0,Contingent liabilities/Net worth,0.043603,0.012742
1,Tax rate (A),0.038215,0.012478
2,Research and development expense rate,0.018083,0.014980
3,Cash Turnover Rate,0.018050,0.006006
4,Fixed Assets Turnover Frequency,0.017917,0.005149
5,Cash/Total Assets,0.013455,0.008687
6,Long-term fund suitability ratio (A),0.012959,0.005424
7,Equity to Long-term Liability,0.012860,0.010826
8,Revenue per person,0.010645,0.006362
9,Fixed Assets to Assets,0.010446,0.006640


## 3. Churn sample-size matched to Taiwan minority count

In [4]:
X_churn, y_churn = _churn_c2st_features(churn_df)
taiwan_minority_n = int(y_taiwan.sum())

churn_matched_rows = []
for boot_i in range(N_BOOT):
    boot_seed = SEED + boot_i
    X_churn_sub, y_churn_sub = _subsample_positive_class(
        X_churn, y_churn, n_positive=taiwan_minority_n, seed=boot_seed
    )
    result = _run_task_pair(X_churn_sub, y_churn_sub, seed=boot_seed)
    churn_matched_rows.append(
        {
            "boot": boot_i,
            "seed": boot_seed,
            "auc": result["smote_auc"]["mean"],
            "sanity_floor_auc": result["dup_auc"]["mean"],
            "positive_rate": result["task_balance"],
            "n_real": result["smote_meta"]["n_real"],
            "n_features": result["smote_meta"]["n_features"],
        }
    )
    print(
        f"churn@taiwan_n boot {boot_i + 1:02d}/{N_BOOT}: "
        f"AUC={result['smote_auc']['mean']:.6f}, sanity={result['dup_auc']['mean']:.6f}"
    )

churn_matched_df = pd.DataFrame(churn_matched_rows)
display(churn_matched_df.head())

churn@taiwan_n boot 01/20: AUC=0.670265, sanity=0.500000


churn@taiwan_n boot 02/20: AUC=0.631663, sanity=0.500000


churn@taiwan_n boot 03/20: AUC=0.655183, sanity=0.500000


churn@taiwan_n boot 04/20: AUC=0.617975, sanity=0.500000


churn@taiwan_n boot 05/20: AUC=0.613378, sanity=0.500000


...

churn@taiwan_n boot 17/20: AUC=0.642424, sanity=0.500000


churn@taiwan_n boot 18/20: AUC=0.614015, sanity=0.500000


churn@taiwan_n boot 19/20: AUC=0.607679, sanity=0.500000


churn@taiwan_n boot 20/20: AUC=0.523760, sanity=0.500000


,boot,seed,auc,sanity_floor_auc,positive_rate,n_real,n_features
0,0,42,0.670265,0.5,0.5,220,40
1,1,43,0.631663,0.5,0.5,220,40
2,2,44,0.655183,0.5,0.5,220,40
3,3,45,0.617975,0.5,0.5,220,40
4,4,46,0.613378,0.5,0.5,220,40


## 4. Taiwan feature-count matched to churn

In [5]:
taiwan_feature_rows = []
taiwan_feature_columns: dict[str, list[str]] = {}
for boot_i in range(N_BOOT):
    boot_seed = SEED + boot_i
    rng = np.random.default_rng(boot_seed)
    columns = rng.choice(X_taiwan.columns.to_numpy(), size=CHURN_FEATURE_COUNT, replace=False)
    subset_columns = [str(col) for col in columns]
    taiwan_feature_columns[str(boot_i)] = subset_columns
    result = _run_task_pair(X_taiwan[subset_columns], y_taiwan, seed=boot_seed)
    taiwan_feature_rows.append(
        {
            "boot": boot_i,
            "seed": boot_seed,
            "auc": result["smote_auc"]["mean"],
            "sanity_floor_auc": result["dup_auc"]["mean"],
            "positive_rate": result["task_balance"],
            "n_real": result["smote_meta"]["n_real"],
            "n_features": result["smote_meta"]["n_features"],
        }
    )
    print(
        f"taiwan@churn_features boot {boot_i + 1:02d}/{N_BOOT}: "
        f"AUC={result['smote_auc']['mean']:.6f}, sanity={result['dup_auc']['mean']:.6f}"
    )

taiwan_feature_df = pd.DataFrame(taiwan_feature_rows)
display(taiwan_feature_df.head())

taiwan@churn_features boot 01/20: AUC=0.618010, sanity=0.500000


taiwan@churn_features boot 02/20: AUC=0.659022, sanity=0.500000


taiwan@churn_features boot 03/20: AUC=0.683161, sanity=0.500000


taiwan@churn_features boot 04/20: AUC=0.611587, sanity=0.500000


taiwan@churn_features boot 05/20: AUC=0.608368, sanity=0.500000


...

taiwan@churn_features boot 17/20: AUC=0.582541, sanity=0.500000


taiwan@churn_features boot 18/20: AUC=0.612707, sanity=0.500000


taiwan@churn_features boot 19/20: AUC=0.606973, sanity=0.500000


taiwan@churn_features boot 20/20: AUC=0.666185, sanity=0.500000


,boot,seed,auc,sanity_floor_auc,positive_rate,n_real,n_features
0,0,42,0.618010,0.5,0.5,220,40
1,1,43,0.659022,0.5,0.5,220,40
2,2,44,0.683161,0.5,0.5,220,40
3,3,45,0.611587,0.5,0.5,220,40
4,4,46,0.608368,0.5,0.5,220,40


## 5. AUC summary and verdict

In [6]:
nb05_report = (OUTPUT_DIR / "05_c2st_verification.md").read_text(encoding="utf-8")
nb05_auc = _read_json_block_after_heading(nb05_report, "## C2ST AUC table")
nb05_metadata = _read_json_block_after_heading(nb05_report, "Raw metadata JSON:")
churn_full_summary = {
    "mean_auc": nb05_auc["real_vs_smote"]["mean_auc"],
    "std_auc": nb05_auc["real_vs_smote"]["std_auc"],
    "ci_low": nb05_auc["real_vs_smote"]["ci_low"],
    "ci_high": nb05_auc["real_vs_smote"]["ci_high"],
    "n_runs": nb05_auc["real_vs_smote"]["n_folds"],
}
churn_full_sanity = nb05_auc["real_vs_duplicated"]["mean_auc"]

taiwan_full_summary = _single_auc_summary(taiwan_full["smote_auc"])
churn_matched_summary = _summary_from_values(churn_matched_df["auc"].tolist())
taiwan_feature_summary = _summary_from_values(taiwan_feature_df["auc"].tolist())

if churn_matched_summary["ci_low"] > taiwan_feature_summary["ci_high"]:
    demarcation_supported = "yes"
elif taiwan_feature_summary["ci_low"] > churn_matched_summary["ci_high"]:
    demarcation_supported = "no"
else:
    demarcation_supported = "inconclusive"

verdict_rule = (
    "yes if churn@taiwan_n 95% interval is entirely above "
    "taiwan@churn_features; no if Taiwan interval is entirely above churn; "
    "inconclusive otherwise"
)

auc_table = pd.DataFrame(
    [
        {
            "condition": "churn_full_from_nb05",
            **churn_full_summary,
            "sanity_floor_auc": churn_full_sanity,
        },
        {
            "condition": "taiwan_full",
            **taiwan_full_summary,
            "sanity_floor_auc": taiwan_full["dup_auc"]["mean"],
        },
        {
            "condition": "churn@taiwan_n",
            **churn_matched_summary,
            "sanity_floor_auc": float(churn_matched_df["sanity_floor_auc"].mean()),
        },
        {
            "condition": "taiwan@churn_features",
            **taiwan_feature_summary,
            "sanity_floor_auc": float(taiwan_feature_df["sanity_floor_auc"].mean()),
        },
    ]
).set_index("condition")
display(auc_table)
print(f"demarcation_supported={demarcation_supported}")
print(verdict_rule)

,mean_auc,std_auc,ci_low,ci_high,n_runs,sanity_floor_auc
condition,,,,,,
churn_full_from_nb05,0.860849,0.007294,0.857158,0.864540,15,0.5
taiwan_full,0.725551,0.058360,0.696017,0.755085,15,0.5
churn@taiwan_n,0.618255,0.032430,0.549121,0.673185,20,0.5
taiwan@churn_features,0.611051,0.051866,0.507384,0.694487,20,0.5


demarcation_supported=inconclusive
yes if churn@taiwan_n 95% interval is entirely above taiwan@churn_features; no if Taiwan interval is entirely above churn; inconclusive otherwise


## 6. Interpretation

In [7]:
importance_top5 = taiwan_top_importances.head(5)["feature"].tolist()
importance_share_top5 = float(
    taiwan_top_importances.head(5)["importance_mean"].sum()
    / taiwan_top_importances["importance_mean"].clip(lower=0).sum()
)

if demarcation_supported == "yes":
    verdict_sentence = (
        "The controlled intervals separate with churn higher than Taiwan, so this supports "
        "off-manifold synthesis as a candidate demarcation mechanism: synthetic points are "
        "more detectable on the panel data where SMOTE harms ranking."
    )
elif demarcation_supported == "no":
    verdict_sentence = (
        "The controlled intervals separate with Taiwan higher than churn, so off-manifold "
        "detectability alone does not explain why SMOTE is harmful on churn but neutral on Taiwan."
    )
else:
    verdict_sentence = (
        "The controlled intervals overlap, so this probe is inconclusive: off-manifold "
        "detectability may matter, but this run does not cleanly separate the panel and "
        "cross-sectional cases once sample size and feature count are controlled."
    )

interpretation = f"""
**Naive comparison.** Notebook 05's churn-full C2ST AUC was {churn_full_summary['mean_auc']:.4f}; Taiwan-full is {taiwan_full_summary['mean_auc']:.4f}. This comparison is useful context but not apples-to-apples because churn has many more minority examples and Taiwan has more features.

**Sample-size control.** When churn is reduced to Taiwan's minority count ({taiwan_minority_n:,}), churn@taiwan_n has mean AUC {churn_matched_summary['mean_auc']:.4f} with empirical 95% interval [{churn_matched_summary['ci_low']:.4f}, {churn_matched_summary['ci_high']:.4f}]. This removes the minority-sample-size advantage from churn.

**Feature-count control.** When Taiwan is restricted to {CHURN_FEATURE_COUNT} random features, Taiwan@churn_features has mean AUC {taiwan_feature_summary['mean_auc']:.4f} with empirical 95% interval [{taiwan_feature_summary['ci_low']:.4f}, {taiwan_feature_summary['ci_high']:.4f}]. This removes Taiwan's larger feature-count advantage.

**Machine-readable verdict.** `demarcation_supported={demarcation_supported}` using the stated interval-separation rule. {verdict_sentence}

**Taiwan importances.** The top Taiwan C2ST features are {', '.join(importance_top5)}. The top five account for {importance_share_top5:.1%} of the positive top-20 permutation importance mass, so detectability is {'concentrated' if importance_share_top5 >= 0.5 else 'fairly diffuse'} rather than the churn result's qualitatively TAG-heavy panel-lag pattern.

**Scope.** C2ST measures whether synthetic minority points are distinguishable from real minority points. It does not measure ranking harm directly. Notebook 03 remains the Taiwan ranking evidence, notebook 02 remains the churn ranking evidence, and this notebook only probes whether point-level detectability helps explain the harmful-vs-neutral demarcation.
"""

display(Markdown(interpretation))


**Naive comparison.** Notebook 05's churn-full C2ST AUC was 0.8608; Taiwan-full is 0.7256. This comparison is useful context but not apples-to-apples because churn has many more minority examples and Taiwan has more features.

**Sample-size control.** When churn is reduced to Taiwan's minority count (220), churn@taiwan_n has mean AUC 0.6183 with empirical 95% interval [0.5491, 0.6732]. This removes the minority-sample-size advantage from churn.

**Feature-count control.** When Taiwan is restricted to 40 random features, Taiwan@churn_features has mean AUC 0.6111 with empirical 95% interval [0.5074, 0.6945]. This removes Taiwan's larger feature-count advantage.

**Machine-readable verdict.** `demarcation_supported=inconclusive` using the stated interval-separation rule. The controlled intervals overlap, so this probe is inconclusive: off-manifold detectability may matter, but this run does not cleanly separate the panel and cross-sectional cases once sample size and feature count are controlled.

**Taiwan importances.** The top Taiwan C2ST features are  Contingent liabilities/Net worth,  Tax rate (A),  Research and development expense rate,  Cash Turnover Rate,  Fixed Assets Turnover Frequency. The top five account for 52.0% of the positive top-20 permutation importance mass, so detectability is concentrated rather than the churn result's qualitatively TAG-heavy panel-lag pattern.

**Scope.** C2ST measures whether synthetic minority points are distinguishable from real minority points. It does not measure ranking harm directly. Notebook 03 remains the Taiwan ranking evidence, notebook 02 remains the churn ranking evidence, and this notebook only probes whether point-level detectability helps explain the harmful-vs-neutral demarcation.


## 7. Verification report

In [8]:
taiwan_sha256_12 = hashlib.sha256(taiwan_path.read_bytes()).hexdigest()[:12]
churn_sha256_12 = hashlib.sha256(RAW_CHURN_PATH.read_bytes()).hexdigest()[:12]

all_sanity_values = [
    churn_full_sanity,
    taiwan_full["dup_auc"]["mean"],
    *churn_matched_df["sanity_floor_auc"].tolist(),
    *taiwan_feature_df["sanity_floor_auc"].tolist(),
]
all_balance_values = [
    taiwan_full["task_balance"],
    taiwan_full["dup_balance"],
    *churn_matched_df["positive_rate"].tolist(),
    *taiwan_feature_df["positive_rate"].tolist(),
]
reported_values_present = all(
    np.isfinite(value)
    for value in [
        taiwan_full_summary["mean_auc"],
        taiwan_full_summary["std_auc"],
        churn_matched_summary["mean_auc"],
        churn_matched_summary["std_auc"],
        churn_matched_summary["ci_low"],
        churn_matched_summary["ci_high"],
        taiwan_feature_summary["mean_auc"],
        taiwan_feature_summary["std_auc"],
        taiwan_feature_summary["ci_low"],
        taiwan_feature_summary["ci_high"],
    ]
)

checks = [
    {
        "check": "Every real-vs-duplicated sanity floor AUC in [0.47, 0.53]",
        "measured": f"min={min(all_sanity_values):.6f}, max={max(all_sanity_values):.6f}",
        "target": "all in [0.47, 0.53]",
        "result": "PASS" if all(0.47 <= v <= 0.53 for v in all_sanity_values) else "FAIL",
    },
    {
        "check": "Every C2ST task class-balanced in [0.45, 0.55]",
        "measured": f"min={min(all_balance_values):.6f}, max={max(all_balance_values):.6f}",
        "target": "all in [0.45, 0.55]",
        "result": "PASS" if all(0.45 <= v <= 0.55 for v in all_balance_values) else "FAIL",
    },
    {
        "check": "Taiwan full and matched-condition AUC summaries are reported",
        "measured": (
            f"taiwan_full={taiwan_full_summary['mean_auc']:.6f}; "
            f"churn@taiwan_n={churn_matched_summary['mean_auc']:.6f}+/-{churn_matched_summary['std_auc']:.6f}; "
            f"taiwan@churn_features={taiwan_feature_summary['mean_auc']:.6f}+/-{taiwan_feature_summary['std_auc']:.6f}"
        ),
        "target": "reported; no PASS/FAIL on AUC values",
        "result": "PASS" if reported_values_present else "FAIL",
    },
    {
        "check": "Machine-readable verdict field is valid",
        "measured": demarcation_supported,
        "target": '{"yes", "no", "inconclusive"}',
        "result": "PASS" if demarcation_supported in {"yes", "no", "inconclusive"} else "FAIL",
    },
]
checks_table = pd.DataFrame(checks)
for check in checks:
    print(
        f"[{check['result']}] {check['check']} | measured: {check['measured']} | "
        f"target: {check['target']}"
    )

[PASS] Every real-vs-duplicated sanity floor AUC in [0.47, 0.53] | measured: min=0.500000, max=0.500000 | target: all in [0.47, 0.53]
[PASS] Every C2ST task class-balanced in [0.45, 0.55] | measured: min=0.500000, max=0.500000 | target: all in [0.45, 0.55]
[PASS] Taiwan full and matched-condition AUC summaries are reported | measured: taiwan_full=0.725551; churn@taiwan_n=0.618255+/-0.032430; taiwan@churn_features=0.611051+/-0.051866 | target: reported; no PASS/FAIL on AUC values
[PASS] Machine-readable verdict field is valid | measured: inconclusive | target: {"yes", "no", "inconclusive"}


In [9]:
c2st_source = inspect.getsource(c2st_module)
auc_report_json = json.dumps(auc_table.to_dict(orient="index"), indent=2)
taiwan_importance_json = json.dumps(taiwan_top_importances.to_dict(orient="records"), indent=2)
churn_matched_json = json.dumps(churn_matched_df.to_dict(orient="records"), indent=2)
taiwan_feature_json = json.dumps(taiwan_feature_df.to_dict(orient="records"), indent=2)
verdict_json = json.dumps(
    {"demarcation_supported": demarcation_supported, "rule": verdict_rule}, indent=2
)

report_lines = [
    "# 06 ? Taiwan C2ST Demarcation Probe Verification Report",
    "",
    "## Environment",
    f"- Python: {platform.python_version()}",
    f"- numpy: {np.__version__}",
    f"- pandas: {pd.__version__}",
    f"- scikit-learn: {sklearn.__version__}",
    f"- imbalanced-learn: {imblearn.__version__}",
    f"- scipy: {scipy.__version__}",
    f"- matplotlib: {matplotlib.__version__}",
    f"- Seed: {SEED}",
    "",
    "## CV config",
    f"- n_splits: {N_SPLITS}",
    f"- n_repeats: {N_REPEATS}",
    f"- n_boot: {N_BOOT}",
    "- Classifier: src.models.factory.make_gbm",
    "- SMOTE: imblearn.over_sampling.SMOTE with default k_neighbors unless minority n requires lowering",
    "",
    "## Dataset fingerprints and C2ST task sizes",
    f"- Taiwan source: {taiwan_path}",
    f"- Taiwan sha256(first 12): {taiwan_sha256_12}",
    f"- Taiwan rows/features: {len(X_taiwan):,} / {X_taiwan.shape[1]}",
    f"- Taiwan positives: {int(y_taiwan.sum()):,}",
    f"- Churn source: {RAW_CHURN_PATH}",
    f"- Churn sha256(first 12): {churn_sha256_12}",
    f"- Churn C2ST feature count: {CHURN_FEATURE_COUNT}",
    f"- Churn full C2ST n_real from nb05: {nb05_metadata['real_vs_smote']['n_real']:,}",
    f"- Churn full C2ST n_smote_synthetic_total from nb05: {nb05_metadata['real_vs_smote']['n_smote_synthetic_total']:,}",
    f"- Churn full C2ST n_smote_synthetic_used from nb05: {nb05_metadata['real_vs_smote']['n_smote_synthetic_used']:,}",
    f"- Churn full task positive_rate from nb05: {nb05_metadata['real_vs_smote']['positive_rate']:.6f}",
    f"- Taiwan full C2ST n_real: {taiwan_full['smote_meta']['n_real']}",
    f"- Taiwan full C2ST n_smote_synthetic_total: {taiwan_full['smote_meta']['n_smote_synthetic_total']}",
    f"- Taiwan full task positive_rate: {taiwan_full['task_balance']:.6f}",
    "",
    "## AUC table",
    "",
    _df_to_md_table(auc_table),
    "",
    "Raw JSON:",
    "```json",
    auc_report_json,
    "```",
    "",
    "## Matched-condition per-boot records",
    "",
    "### Churn at Taiwan minority n",
    "```json",
    churn_matched_json,
    "```",
    "",
    "### Taiwan at churn feature count",
    "```json",
    taiwan_feature_json,
    "```",
    "",
    "## Taiwan top permutation importances",
    "",
    _df_to_md_table(taiwan_top_importances.set_index("feature")),
    "",
    "Raw JSON:",
    "```json",
    taiwan_importance_json,
    "```",
    "",
    "## Machine-readable verdict",
    "```json",
    verdict_json,
    "```",
    "",
    "## Interpretation",
    interpretation.strip(),
    "",
    "## Inline source",
    "",
    "### src/evaluation/c2st.py",
    "```python",
    c2st_source,
    "```",
    "",
    "## Self-run acceptance checks",
    _df_to_md_table(checks_table.set_index("check"), float_format="{}"),
    "",
]

report_text = "\n".join(report_lines)
report_path = OUTPUT_DIR / "06_c2st_taiwan_verification.md"
report_path.write_text(report_text, encoding="utf-8")
print(report_text)

# 06 ? Taiwan C2ST Demarcation Probe Verification Report

## Environment
- Python: 3.12.3
- numpy: 2.4.6
- pandas: 3.0.3
- scikit-learn: 1.9.0
- imbalanced-learn: 0.14.2
- scipy: 1.18.0
- matplotlib: 3.11.0
- Seed: 42

## CV config
- n_splits: 5
- n_repeats: 3
- n_boot: 20
- Classifier: src.models.factory.make_gbm
- SMOTE: imblearn.over_sampling.SMOTE with default k_neighbors unless minority n requires lowering

## Dataset fingerprints and C2ST task sizes
- Taiwan source: C:\Users\hpdk1\Documents\Projects\thesis-recreation-2026\data\raw\taiwan_bankruptcy.csv
- Taiwan sha256(first 12): cbe49e73a9df
- Taiwan rows/features: 6,819 / 95
- Taiwan positives: 220
- Churn source: C:\Users\hpdk1\Documents\Projects\thesis-recreation-2026\data\raw\training_clean.xlsx
- Churn sha256(first 12): c04f3a55aa4f
- Churn C2ST feature count: 40
- Churn full C2ST n_real from nb05: 3,804
- Churn full C2ST n_smote_synthetic_total from nb05: 92,198
- Churn full C2ST n_smote_synthetic_used from nb05: 3,804
- Ch